In [9]:

from dataclasses import dataclass
from typing import List, Tuple, Optional, Dict
import numpy as np
import copy
from collections import defaultdict, deque
import pickle
import random
from tqdm import trange

In [10]:
# Operator type
Operator = Optional[str]

@dataclass
class Piece:
    player: int  # 1 = Blue, -1 = Red
    value: int
    dama: bool = False

    def __repr__(self):
        p = "🔵" if self.player == 1 else "🔴"
        val = f"{self.value:+d}" if self.value >= 0 else str(self.value)
        return f"{p}{'D' if self.dama else ''}{val}"
    
    def copy(self):
        return Piece(self.player, self.value, self.dama)
    
@dataclass
class Move:
    path: List[Tuple[int, int]]
    captures: List[Tuple[int, int]]
    promotes: bool = False
    score_gain: int = 0
    is_dama_capture: bool = False
    is_multi_jump: bool = False

    def __repr__(self):
        cap_str = f" x{len(self.captures)}" if self.captures else ""
        promo = " (promo)" if self.promotes else ""
        return f"{self.path}{cap_str}{promo} +{self.score_gain}"

In [11]:
class DamathEnv:
    def __init__(self, rows=8, cols=8, operator_pattern=None):
        self.R = rows
        self.C = cols
        if operator_pattern is None:
            operator_pattern = self.default_operator_board()
        self.op_board = operator_pattern
        self.pieces: Dict[Tuple[int,int], Piece] = {}
        self.scores = {1: 0.0, -1: 0.0}
        self.to_move = 1
        self.history_states = deque(maxlen=50)
        self.init_default_integer_setup()

    def default_operator_board(self):
        ops = ['x','/','-','+']
        board = [[None for _ in range(self.C)] for __ in range(self.R)]
        for r in range(self.R):
            for c in range(self.C):
                if (r + c) % 2 == 1:
                    board[r][c] = ops[(r + 2*c) % len(ops)]
        return board

    def init_default_integer_setup(self):
        self.pieces = {}
        blue_values = [[-11, 8, -5, 2], [0, -3, 10, -7], [-9, 6, -1, 4]]
        red_values = [[4, -1, 6, -9], [-7, 10, -3, 0], [2, -5, 8, -11]]
        
        def playable_positions_on_row(r):
            return [c for c in range(self.C) if (r+c)%2==1]
        
        for i, r in enumerate(range(0,3)):
            cols = playable_positions_on_row(r)
            vals = blue_values[i]
            for j, c in enumerate(cols[:len(vals)]):
                self.pieces[(r,c)] = Piece(player=1, value=vals[j], dama=False)
        
        for i, r in enumerate(range(5,8)):
            cols = playable_positions_on_row(r)
            vals = red_values[i]
            for j, c in enumerate(cols[:len(vals)]):
                self.pieces[(r,c)] = Piece(player=-1, value=vals[j], dama=False)
        
        self.scores = {1:0.0, -1:0.0}
        self.to_move = 1
        self.history_states.clear()
        self.record_state()

    def copy(self):
        newenv = DamathEnv(self.R, self.C)
        newenv.op_board = copy.deepcopy(self.op_board)
        newenv.pieces = {k: v.copy() for k,v in self.pieces.items()}
        newenv.scores = dict(self.scores)
        newenv.to_move = self.to_move
        newenv.history_states = copy.deepcopy(self.history_states)
        return newenv

    def in_bounds(self, r, c):
        return 0 <= r < self.R and 0 <= c < self.C

    def is_playable(self, r, c):
        return self.in_bounds(r,c) and ((r+c)%2==1)

    def record_state(self):
        items = tuple(sorted([(pos, p.player, p.value, p.dama) for pos, p in self.pieces.items()]))
        key = (self.to_move, items)
        self.history_states.append(key)

    def op_at(self, r, c):
        return self.op_board[r][c] if self.is_playable(r,c) else None

    def apply_operator(self, op: str, a: int, b: int):
        if op == '+': return a + b
        if op == '-': return a - b
        if op in ['x', 'X', '*']: return a * b
        if op == '/': return int(a / b) if b != 0 else 0
        raise ValueError(f"Unknown op {op}")

    def generate_all_moves(self, player: int):
        capture_moves = []
        for pos, piece in list(self.pieces.items()):
            if piece.player != player: continue
            caps = self._generate_captures_from(pos, piece)
            capture_moves.extend(caps)
        
        if capture_moves:
            max_cap = max(len(m.captures) for m in capture_moves)
            maxcap_moves = [m for m in capture_moves if len(m.captures)==max_cap]
            if any(self.pieces[m.path[0]].dama for m in maxcap_moves):
                maxcap_moves = [m for m in maxcap_moves if self.pieces[m.path[0]].dama]
            for m in maxcap_moves:
                m.score_gain = self._compute_move_score(m, player)
            return maxcap_moves
        
        simple_moves = []
        for pos, piece in list(self.pieces.items()):
            if piece.player != player: continue
            sms = self._generate_simple_from(pos, piece)
            simple_moves.extend(sms)
        for m in simple_moves:
            m.score_gain = 0.0
        return simple_moves

    def _generate_simple_from(self, pos, piece: Piece):
        r, c = pos
        moves = []
        if piece.dama:
            for dr, dc in [(-1,-1),(-1,1),(1,-1),(1,1)]:
                step = 1
                while True:
                    nr, nc = r + dr*step, c + dc*step
                    if not self.in_bounds(nr,nc) or not self.is_playable(nr,nc): break
                    if (nr,nc) in self.pieces: break
                    path = [(r,c),(nr,nc)]
                    promotes = self._check_promotion(nr, piece.player)
                    moves.append(Move(path=path, captures=[], promotes=promotes))
                    step += 1
        else:
            dr = 1 if piece.player==1 else -1
            for dc in (-1,1):
                nr, nc = r + dr, c + dc
                if not self.in_bounds(nr,nc) or not self.is_playable(nr,nc): continue
                if (nr,nc) in self.pieces: continue
                path = [(r,c),(nr,nc)]
                promotes = self._check_promotion(nr, piece.player)
                moves.append(Move(path=path, captures=[], promotes=promotes))
        return moves

    def _generate_captures_from(self, pos, piece: Piece):
        results = []
        r, c = pos
        
        if not piece.dama:
            for dr, dc in [(-1,-1),(-1,1),(1,-1),(1,1)]:
                ar, ac = r + dr, c + dc
                lr, lc = r + 2*dr, c + 2*dc
                if not self.in_bounds(ar,ac) or not self.is_playable(ar,ac): continue
                if (ar,ac) not in self.pieces: continue
                if self.pieces[(ar,ac)].player == piece.player: continue
                if not self.in_bounds(lr,lc) or not self.is_playable(lr,lc): continue
                if (lr,lc) in self.pieces: continue
                
                new_env = self.copy()
                captured_piece = new_env.pieces.pop((ar,ac))
                moved_piece = new_env.pieces.pop((r,c))
                new_env.pieces[(lr,lc)] = moved_piece
                further = new_env._generate_captures_from((lr,lc), moved_piece)
                
                if not further:
                    m = Move(path=[(r,c),(lr,lc)], captures=[(ar,ac,captured_piece)], 
                            promotes=self._check_promotion(lr,piece.player))
                    results.append(m)
                else:
                    for fm in further:
                        m = Move(path=[(r,c)] + fm.path, 
                                captures=[(ar,ac,captured_piece)] + fm.captures,
                                promotes=self._check_promotion(fm.path[-1][0], piece.player))
                        results.append(m)
        
        return results

    def _compute_move_score(self, move: Move, mover_player: int):
        total = 0.0
        mover_start = move.path[0]
        mover_piece = self.pieces.get(mover_start)
        mover_is_dama = mover_piece.dama if mover_piece else False
        mover_value = mover_piece.value if mover_piece else 0
        
        for i, (cap_r, cap_c, cap_piece) in enumerate(move.captures):
            landing = move.path[1+i] if len(move.path) > 1+i else move.path[-1]
            op = self.op_at(*landing)
            base = self.apply_operator(op, mover_value, cap_piece.value)
            
            mult = 1
            if mover_is_dama and cap_piece.dama:
                mult = 4
            elif mover_is_dama or cap_piece.dama:
                mult = 2
            total += base * mult
        return total

    def _check_promotion(self, r, player):
        if player==1 and r==self.R-1: return True
        if player==-1 and r==0: return True
        return False

    def apply_move(self, move: Move):
        player = self.to_move
        total_gain = 0.0
        
        if not move.captures:
            frm, to = move.path[0], move.path[-1]
            piece = self.pieces.pop(frm)
            self.pieces[to] = piece
            if self._check_promotion(to[0], piece.player) and not piece.dama:
                piece.dama = True
        else:
            frm = move.path[0]
            mover = self.pieces.pop(frm)
            current_pos = frm
            
            for i, (cap_r, cap_c, cap_piece_snapshot) in enumerate(move.captures):
                landing = move.path[1 + i] if 1 + i < len(move.path) else move.path[-1]
                op = self.op_at(*landing)
                self.pieces.pop((cap_r, cap_c))
                
                base = self.apply_operator(op, mover.value, cap_piece_snapshot.value)
                mult = 1
                if mover.dama and cap_piece_snapshot.dama:
                    mult = 4
                elif mover.dama or cap_piece_snapshot.dama:
                    mult = 2
                
                gain = base * mult
                total_gain += gain
                current_pos = landing
            
            self.pieces[current_pos] = mover
            if self._check_promotion(current_pos[0], mover.player) and not mover.dama:
                mover.dama = True
            
            self.scores[mover.player] += total_gain
        
        self.to_move *= -1
        self.record_state()
        return total_gain

    def game_over(self):
        if not self.generate_all_moves(self.to_move):
            return True
        players_present = set(p.player for p in self.pieces.values())
        if len(players_present) <= 1:
            return True
        return False

    def final_scores_and_winner(self):
        final_scores = dict(self.scores)
        for pos, piece in self.pieces.items():
            val = piece.value * (2 if piece.dama else 1)
            final_scores[piece.player] += val
        
        if final_scores[1] > final_scores[-1]:
            winner = 1
        elif final_scores[1] < final_scores[-1]:
            winner = -1
        else:
            winner = 0
        return final_scores, winner

    def get_state_hash(self):
        items = tuple(sorted([(pos, p.player, p.value, p.dama) 
                             for pos, p in self.pieces.items()]))
        return (self.to_move, items)

In [12]:
class QLearner:
    def __init__(self, alpha=0.1, gamma=0.95, epsilon=1.0, min_epsilon=0.05, decay=0.9995):
        self.Q = defaultdict(lambda: defaultdict(float))
        self.alpha = alpha
        self.gamma = gamma
        self.epsilon = epsilon
        self.min_epsilon = min_epsilon
        self.decay = decay

    def get_q(self, state, action):
        return self.Q[state][action]

    def choose_action(self, env, moves):
        if not moves:
            return None
        
        if random.random() < self.epsilon:
            return random.choice(moves)
        
        state = env.get_state_hash()
        q_values = [(self.get_q(state, self._move_to_key(m)), m) for m in moves]
        max_q = max(q_values, key=lambda x: x[0])[0]
        best_moves = [m for q, m in q_values if q == max_q]
        return random.choice(best_moves)

    def update(self, state, action, reward, next_state, next_moves, done):
        state_key = state
        action_key = self._move_to_key(action)
        
        current_q = self.get_q(state_key, action_key)
        
        if done or not next_moves:
            target = reward
        else:
            next_state_key = next_state
            max_next_q = max([self.get_q(next_state_key, self._move_to_key(m)) 
                             for m in next_moves], default=0.0)
            target = reward + self.gamma * max_next_q
        
        self.Q[state_key][action_key] = current_q + self.alpha * (target - current_q)

    def decay_epsilon(self):
        self.epsilon = max(self.min_epsilon, self.epsilon * self.decay)

    def _move_to_key(self, move):
        return (tuple(move.path), tuple((r, c) for r, c, _ in move.captures))

    def save(self, path):
        with open(path, 'wb') as f:
            pickle.dump(dict(self.Q), f)

    def load(self, path):
        with open(path, 'rb') as f:
            self.Q = defaultdict(lambda: defaultdict(float), pickle.load(f))


In [13]:
def train_qlearning_selfplay(episodes=1000, verbose_every=250):
    """Train Q-Learning agent through self-play"""
    agent = QLearner(alpha=0.15, gamma=0.95, epsilon=1.0, min_epsilon=0.05, decay=0.9995)
    
    blue_wins = red_wins = draws = 0
    total_blue_score = 0
    total_red_score = 0
    
    for ep in trange(episodes, desc="Self-Play Training"):
        env = DamathEnv()
        experiences = {1: [], -1: []}  # Store experiences for both players
        
        while not env.game_over():
            player = env.to_move
            moves = env.generate_all_moves(player)
            
            if not moves:
                break
            
            state = env.get_state_hash()
            
            # Agent plays both sides
            move = agent.choose_action(env, moves)
            
            reward = env.apply_move(move)
            next_state = env.get_state_hash()
            next_moves = env.generate_all_moves(env.to_move) if not env.game_over() else []
            done = env.game_over()
            
            # Store experience for current player
            experiences[player].append((state, move, reward, next_state, next_moves, done))
        
        # Game ended
        final_scores, winner = env.final_scores_and_winner()
        
        # Update Q-table for both players' moves
        for player in [1, -1]:
            if winner == player:
                outcome_reward = 10.0
            elif winner == -player:
                outcome_reward = -10.0
            else:
                outcome_reward = 0.0
            
            # Update all experiences for this player
            for i, (state, action, step_reward, next_state, next_moves, done) in enumerate(experiences[player]):
                is_last = (i == len(experiences[player]) - 1)
                total_reward = step_reward + (outcome_reward if is_last else 0)
                agent.update(state, action, total_reward, next_state, next_moves, done)
        
        agent.decay_epsilon()
        
        # Track statistics
        total_blue_score += final_scores[1]
        total_red_score += final_scores[-1]
        
        if winner == 1:
            blue_wins += 1
        elif winner == -1:
            red_wins += 1
        else:
            draws += 1
        
        if (ep + 1) % verbose_every == 0:
            avg_blue_score = total_blue_score / verbose_every
            avg_red_score = total_red_score / verbose_every
            
            print(f"\nEpisode {ep + 1}:")
            print(f"  Blue wins: {blue_wins}, Red wins: {red_wins}, Draws: {draws}")
            print(f"  Avg Scores - Blue: {avg_blue_score:.1f}, Red: {avg_red_score:.1f}")
            print(f"  Epsilon: {agent.epsilon:.4f}, Q-table size: {len(agent.Q)}")
            
            blue_wins = red_wins = draws = 0
            total_blue_score = 0
            total_red_score = 0
    
    return agent

In [14]:
def evaluate_agent(agent, num_games=100):
    """Evaluate trained agent against random opponent"""
    agent.epsilon = 0.0  # No exploration during evaluation
    
    wins = losses = draws = 0
    total_agent_score = 0
    total_opponent_score = 0
    
    print(f"\n{'='*50}")
    print(f"Evaluating agent over {num_games} games...")
    print(f"{'='*50}")
    
    for game in trange(num_games, desc="Evaluation"):
        env = DamathEnv()
        
        while not env.game_over():
            player = env.to_move
            moves = env.generate_all_moves(player)
            
            if not moves:
                break
            
            if player == 1:  # Agent
                move = agent.choose_action(env, moves)
            else:  # Random opponent
                move = random.choice(moves)
            
            env.apply_move(move)
        
        final_scores, winner = env.final_scores_and_winner()
        total_agent_score += final_scores[1]
        total_opponent_score += final_scores[-1]
        
        if winner == 1:
            wins += 1
        elif winner == -1:
            losses += 1
        else:
            draws += 1
    
    win_rate = wins / num_games * 100
    avg_agent_score = total_agent_score / num_games
    avg_opp_score = total_opponent_score / num_games
    
    print(f"\n{'='*50}")
    print("EVALUATION RESULTS:")
    print(f"{'='*50}")
    print(f"Win Rate: {win_rate:.1f}% ({wins}/{num_games})")
    print(f"Losses: {losses}, Draws: {draws}")
    print(f"Average Score - Agent: {avg_agent_score:.1f}")
    print(f"Average Score - Random: {avg_opp_score:.1f}")
    print(f"Score Margin: {avg_agent_score - avg_opp_score:+.1f}")
    print(f"{'='*50}\n")

In [15]:
# Train the agent against random opponent
print("Starting Q-Learning training against self...")
print("Agent plays as Blue (Player 1), Agent also plays as Red (Player -1)\n")

trained_agent = train_qlearning_selfplay(episodes=1000, verbose_every=250)

print("\n" + "="*50)
print("Training complete!")
print("="*50)

# Evaluate the trained agent
evaluate_agent(trained_agent, num_games=100)

Starting Q-Learning training against random opponent...
Agent plays as Blue (Player 1), Random plays as Red (Player -1)



Self-Play Training:  25%|██▌       | 251/1000 [03:08<02:21,  5.29it/s]  


Episode 250:
  Blue wins: 134, Red wins: 116, Draws: 0
  Avg Scores - Blue: 3.8, Red: -7.0
  Epsilon: 0.8825, Q-table size: 535920


Self-Play Training:  50%|█████     | 500/1000 [04:03<01:57,  4.27it/s]


Episode 500:
  Blue wins: 133, Red wins: 117, Draws: 0
  Avg Scores - Blue: 8.8, Red: 4.3
  Epsilon: 0.7788, Q-table size: 566551


...


Episode 750:
  Blue wins: 110, Red wins: 139, Draws: 1
  Avg Scores - Blue: -2.1, Red: 1.5
  Epsilon: 0.6872, Q-table size: 631846


Self-Play Training: 100%|██████████| 1000/1000 [09:16<00:00,  1.80it/s] 



Episode 1000:
  Blue wins: 129, Red wins: 120, Draws: 1
  Avg Scores - Blue: 13.3, Red: 2.7
  Epsilon: 0.6065, Q-table size: 1286286

Training complete!

Evaluating agent over 100 games...


Evaluation: 100%|██████████| 100/100 [00:10<00:00,  9.43it/s]


EVALUATION RESULTS:
Win Rate: 42.0% (42/100)
Losses: 58, Draws: 0
Average Score - Agent: -0.2
Average Score - Random: 18.1
Score Margin: -18.3



In [16]:
# # Save trained agent
# trained_agent.save('damath_agent_trained.pkl')
# print("✅ Agent saved as 'damath_agent_trained.pkl'")